# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 4,000,000 new records
write_files: 4,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-111156-706b25

Lake check — topic 'logs' (4 partitions)   2026-11-25 11:13:49
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0         [0, 1,034,877)   1,034,877           0          0           0
        1           [0, 993,200)     993,200           0          0           0
        2           [0, 971,202)     971,202           0          0           0
        3         [0, 1,000,721)   1,000,721           0          0           0
-------------------------------------------------------------------------------
    total      4,000,000 records   4,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 222.9 MB, 55.73 MB pe

{'topic': 4000000,
 'lake': 4000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 222.922233,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 4,000,000 new records
write_files: 4,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-111349-b74f60

Lake check — topic 'logs' (4 partitions)   2026-11-25 11:14:35
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0         [0, 1,034,877)   2,069,754   1,034,877          0           0
        1           [0, 993,200)   1,986,400     993,200          0           0
        2           [0, 971,202)   1,942,404     971,202          0           0
        3         [0, 1,000,721)   2,001,442   1,000,721          0           0
-------------------------------------------------------------------------------
    total      4,000,000 records   8,000,000   4,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 445.8 MB, 55.73 MB per file on average
check took 0.4 s

Exa

{'topic': 4000000,
 'lake': 8000000,
 'dup': 4000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 445.844466,
 'ok': False,
 'problems': ['4,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [ ]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    run = write_files(new, STAGING)
    publish(run)
    
    # 4. remember where you stopped (if your architecture needs it)
    ...

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [9]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             4,000,000           0        0           0  41.5s  PASS
2 new records arrive, run again         4,100,000           0        0           0   4.8s  PASS
3 nothing new, run again                4,100,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  4,100,000           0        0     100,000         PASS
  ... then the next run                 4,200,000           0        0           0   4.3s  PASS
5 run stops AFTER writing: right after  4,200,000           0        0     100,000         PASS
  ... then the next run                 4,300,000           0        0           0   3.9s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 4000000,
   'lake': 4000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 222.922233,
   'ok': True,
   'problems': []},
  41.500938415527344),
 ('2 new records arrive, run again',
  {'topic': 4100000,
   'lake': 4100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 224.081961,
   'ok': True,
   'problems': []},
  4.80363917350769),
 ('3 nothing new, run again',
  {'topic': 4100000,
   'lake': 4100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 224.081961,
   'ok': True,
   'problems': []},
  0.059888362884521484),
 ('4 run stops WHILE writing: right after',
  {'topic': 4200000,
   'lake': 4100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 224.081961,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 4200000,
   'lake': 4200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 225.240031,
   'ok': True,
   'problems': []},